# Set Up pwd and auto updates

In [ ]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)


# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

%pwd


# Ensure Data Exists 

In [ ]:
!python collect_external_data/expected_counts.py
!python collect_external_data/road_geom.py

# Exparimentation with season functionality

In [ ]:
from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from season.season_orchestrator import SeasonOrchestrator

import numpy as np
import pandas as pd
from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform


# Creating Schedules for TM 

In [ ]:
traffic_percentile_schedule = ScheduleSpecs(
     mode='static',
    value=50,  
    dist=None
)

bus_interval_schedule = ScheduleSpecs(
    mode='static',
    value=30,  # static bus interval of 15 minutes
    dist=None
)

crashes_schedule = ScheduleSpecs(
    mode='static',
    value=4,  # static bus interval of 15 minutes
    dist=None # normal distribution for crashes per 100k VMT
)


print(traffic_percentile_schedule.realize( n_days=15))
print(crashes_schedule.realize( n_days=15))

# Defining the Population Perams 

for small runs: 
- PopulationParams.population_size == make_season_config.max_persons & small n 
- 


In [ ]:
pop_params = PopulationParams(
    population_size=200,
    value_of_time=lognorm(s=0.64, scale=40/60),
    experience_weight_car=1.0,
    experience_weight_bus=skewnorm(6, loc=1.15, scale=0.3),
    prior_car=22.0,
    prior_bus=30.0,
    time_decay_rate=0.1,
    prior_weight=1.0,
    uncertainty_multiplier=1.0,
    travel_propensity=1.0,
)


config = make_season_config(
    season_id='two_week_example_001',
    run_description='Example config with custom schedules',
    seed=123,
    n_days=14,
    max_steps=10000,
    max_persons=200,
    collect_every_n=1000,
    start_hr=8,
    bus_capacity=30,
    road_path='data/roads/hw210_sl_and_curvs.parquet',
    ecs_path='data/vehicle_counts/expected_counts_seconds.csv',
    toll_mechanism='static',
    toll_params={'car': 20.0, 'bus': 0.0},
    canyon_closures_schedule=None,
    traffic_percentile_schedule=traffic_percentile_schedule,
    bus_interval_schedule=bus_interval_schedule,
    crashes_schedule=crashes_schedule, 
    population_params=pop_params
   
)

# light_config = make_season_config(
#     season_id='light_season_001',
#     run_description='Light config for testing',
#     seed=123,
#     n_days=15,
#     max_steps=3000,
#     max_persons=500,
#     collect_every_n=10,
#     road_path='data/roads/hw210_sl_and_curvs.parquet',
#     ecs_path='data/vehicle_counts/expected_counts_seconds.csv',
#     toll_mechanism='static',
#     toll_params={'car': 8.0, 'bus': 0.0},
#     population_params=pop_params,
#     crashes_schedule=crashes_schedule
    
# )

In [ ]:
# Example usage of SeasonOrchestrator with example_config
orchestrator = SeasonOrchestrator(season_config=config, output_dir="data/season_outputs",)
orchestrator.run_season()

